# M8c — Real photographs (reproduction)

Reproduces the headline tables from `docs/insights/m8c_real_photos.md`:
- PMR(_nolabel) baseline per (category, model)
- Comparison vs synthetic-textured PMR (paired delta)
- PMR by (category, label_role) per model
- H7 paired-difference physical − abstract

60 photos: 12 × {ball, car, person, bird, abstract}. Sources: COCO 2017 val + WikiArt.

In [1]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from physical_mode.metrics.pmr import score_rows
from physical_mode.inference.prompts import LABELS_BY_SHAPE
OUT = PROJECT_ROOT / 'outputs'
def latest(prefix):
    cands = sorted(OUT.glob(f'{prefix}_*/predictions.jsonl'))
    return cands[-1]
Q_LBL = pd.read_json(sorted(OUT.glob('m8c_qwen_2*/predictions.jsonl'))[-1], lines=True)
Q_NL  = pd.read_json(sorted(OUT.glob('m8c_qwen_label_free_*/predictions.jsonl'))[-1], lines=True)
L_LBL = pd.read_json(sorted(OUT.glob('m8c_llava_2*/predictions.jsonl'))[-1], lines=True)
L_NL  = pd.read_json(sorted(OUT.glob('m8c_llava_label_free_*/predictions.jsonl'))[-1], lines=True)
for n, df in [('Q_LBL', Q_LBL), ('Q_NL', Q_NL), ('L_LBL', L_LBL), ('L_NL', L_NL)]:
    print(f'{n}: {len(df)} rows')
Q_LBL = score_rows(Q_LBL); Q_NL = score_rows(Q_NL); L_LBL = score_rows(L_LBL); L_NL = score_rows(L_NL)
def role(s, l):
    if l == '_nolabel': return '_nolabel'
    triplet = LABELS_BY_SHAPE.get(s)
    if triplet is None: return l
    p, a, e = triplet
    return 'physical' if l == p else 'abstract' if l == a else 'exotic' if l == e else l
for df in [Q_LBL, L_LBL]:
    df['label_role'] = [role(s, l) for s, l in zip(df['shape'], df['label'])]
print('annotated.')

Q_LBL: 180 rows
Q_NL: 60 rows
L_LBL: 180 rows
L_NL: 60 rows
annotated.


## Headline 1 — PMR(_nolabel) on photos vs synthetic-textured

**Photos REDUCE PMR(_nolabel) for Qwen across all 4 physical categories** (drop range −0.18 to −0.48). LLaVA: car drops, person rises, bird/ball flat. Photos add scene context that distracts from the single-object physics-mode reading.

In [2]:
categories = ['ball','car','person','bird','abstract']
photo_pmr = pd.DataFrame({
    'Qwen photo':   Q_NL.groupby('shape')['pmr'].mean().round(3),
    'LLaVA photo':  L_NL.groupby('shape')['pmr'].mean().round(3),
}).reindex(categories)
print('PMR(_nolabel) on photos:'); print(photo_pmr)
syn_baseline = pd.read_csv(OUT / 'm8c_summary' / 'm8c_synthetic_baseline.csv')
print('\nSynthetic-textured baseline (M8a + M8d):')
print(syn_baseline.to_string(index=False))

PMR(_nolabel) on photos:
          Qwen photo  LLaVA photo
shape                            
ball           0.667        0.500
car            0.500        0.000
person         0.667        0.417
bird           0.417        0.500
abstract       0.500        0.000

Synthetic-textured baseline (M8a + M8d):
model                    source          category  n  pmr_nolabel
 qwen synthetic-textured-circle              ball 20        0.900
 qwen     synthetic-line-circle abstract-baseline 20        0.800
llava synthetic-textured-circle              ball 20        0.450
llava     synthetic-line-circle abstract-baseline 20        0.050
 qwen    synthetic-textured-m8d               car 40        0.975
 qwen    synthetic-textured-m8d            person 40        0.850
 qwen    synthetic-textured-m8d              bird 40        0.875
llava    synthetic-textured-m8d               car 40        0.375
llava    synthetic-textured-m8d            person 40        0.025
llava    synthetic-textured-m8d    

## Headline 2 — PMR by (category × label_role)

H7 paired-difference physical−abstract on photos:
- LLaVA: ball +0.167 ✓, car 0.000 ✗, person −0.250 ✗, bird +0.667 ✓ → 2/4
- Qwen: ball +0.083, car −0.167, person +0.500, bird 0.000 → 2/4

Compare to M8d synthetic LLaVA H7 = 3/3. Photos add noise but don't reverse the H7 signal cleanly.

In [3]:
def pmr_role(df):
    return df.groupby(['shape','label_role'])['pmr'].mean().unstack()[['physical','abstract','exotic']].reindex(categories).round(3)
print('Qwen photo PMR by (category, label_role):'); print(pmr_role(Q_LBL))
print('\nLLaVA photo PMR by (category, label_role):'); print(pmr_role(L_LBL))
print('\n=== H7 paired-difference (physical − abstract) ===')
for name, df in [('Qwen', pmr_role(Q_LBL)), ('LLaVA', pmr_role(L_LBL))]:
    print(f'  {name}:')
    for cat in categories:
        d = df.loc[cat, 'physical'] - df.loc[cat, 'abstract']
        print(f'    {cat:9s}: {d:+.3f}')

Qwen photo PMR by (category, label_role):
label_role  physical  abstract  exotic
shape                                 
ball           0.667     0.583   0.583
car            0.667     0.833   0.500
person         0.750     0.250   0.500
bird           0.750     0.750   0.583
abstract       0.500     0.500   0.417

LLaVA photo PMR by (category, label_role):
label_role  physical  abstract  exotic
shape                                 
ball           0.667     0.500   0.667
car            0.083     0.083   0.333
person         0.333     0.583   0.417
bird           0.833     0.167   0.500
abstract       0.000     0.083   0.167

=== H7 paired-difference (physical − abstract) ===
  Qwen:
    ball     : +0.084
    car      : -0.166
    person   : +0.500
    bird     : +0.000
    abstract : +0.000
  LLaVA:
    ball     : +0.167
    car      : +0.000
    person   : -0.250
    bird     : +0.666
    abstract : -0.083


## Bottom line

1. **Photos do NOT saturate the encoder more than synthetic stim.** They give muddier signals: PMR(_nolabel) drops 18-48 pp for Qwen across all 4 categories.
2. **LLaVA person photo PMR rises +39 pp** vs synthetic — the encoder finally recognizes humans.
3. **H7 partially holds on photos** (LLaVA 2/4, Qwen 2/4) but is noisier than on synthetic (LLaVA 3/3 in M8d).
4. **Visual-saturation hypothesis refined**: behavioral PMR(_nolabel) saturation is the conjunction of encoder representational saturation AND input-context simplicity. Real photos add scene context; synthetic stim is minimal. Same encoder, different stim type → different PMR.